# RLS initialization across observed SNR levels

保留 SNR=30/10/−10 dB 的 heuristic 初始化比較。δ 表示 inverse-covariance 初值 P₀=δI。有 observation noise 時，clean AR coefficients 不是 noisy-process Wiener optimum。

來源：`rlsInitBySnr.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Parameters and SNR-dependent factories

In [ ]:
n_samples, n_trials, seed, lam = 300, 100, 1234, 0.99
grouped_results = {}
for snr_db in (30.0, 10.0, -10.0):
  observed_variance = 1 + 10**(-snr_db / 10)
  params = af.rlsSnrInitParams(snr_db, n_coef=2, sigma_u2=observed_variance, lam=lam)
  print(f"SNR={snr_db:g} dB:", params)
  make_data = partial(data.ar_prediction, n_samples, coefficients=[-0.1, 0.8], variance=1.0, observation_snr_db=snr_db)
  models = {
    "RLS SNR heuristic": partial(af.RLS, filter_order=1, lam=lam, delta=params["delta"]),
    "RLS unit initial P": partial(af.RLS, filter_order=1, lam=lam, delta=1.0),
  }
  grouped_results[snr_db] = training.compare(models, make_data, n_trials=n_trials, seed=seed)

## 3. Learning and coefficient reference caveat

In [ ]:
for snr_db, results in grouped_results.items():
  plot.learning(results, window=11, title=f"Observed AR(2): SNR={snr_db:g} dB")
  plot.show()
  plot.coefficients(results["RLS SNR heuristic"], reference=np.array([-0.1, 0.8]),
                    title="Clean-process AR coefficients are a reference, not the noisy optimum")
  plot.show()
  training.summary(results, tail=100)